# 실험 01: Chat Model과 Message 클래스

## 1. 실험 제목과 목표
- **제목**: 순수 API에서 LangChain Model 객체로의 전환
- **목표**: 기존의 `openai.OpenAI()` 클라이언트 대신 LangChain의 `ChatOpenAI` 래퍼(Wrapper)를 사용해보고, 파이썬 딕셔너리(`{"role": ...}`) 대신 객체 지향적인 Message 클래스(`SystemMessage`, `HumanMessage`)를 사용하는 방법을 익힙니다.

## 2. 실험 개요
1. **비교 실험 1**: 기존 OpenAI API 호출과 LangChain `ChatOpenAI` 호출 코드 비교
2. **비교 실험 2**: 딕셔너리 포맷 vs LangChain Message 객체(`SystemMessage`, `HumanMessage`, `AIMessage`)
3. **실패/주의 케이스**: 존재하지 않는 모델명을 입력했을 때 LangChain에서 어떻게 에러가 발생하는지 확인

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

# 순수 OpenAI API
from openai import OpenAI

# LangChain 컴포넌트
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

## 4. 환경 변수 로드

In [2]:
load_dotenv()

True

## 5. 비교 실험 1: 순수 API vs LangChain 래퍼(Wrapper)
LangChain은 다양한 LLM(OpenAI, Anthropic, Gemini 등)을 똑같은 방법(인터페이스)으로 호출할 수 있게 감싸줍니다.

In [3]:
question = "LangChain이 뭐야? 한 문장으로 대답해줘."

print("=== [1. 순수 OpenAI API 방식] ===")
raw_client = OpenAI()
raw_response = raw_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": question}]
)
print(raw_response.choices[0].message.content)

print("\n=== [2. LangChain ChatOpenAI 방식] ===")
# 모델 객체를 미리 생성해둡니다.
llm = ChatOpenAI(model="gpt-4o-mini")
# .invoke() 메서드로 깔끔하게 호출합니다.
lc_response = llm.invoke(question)
print(lc_response.content)

print("\n-> 결과: 코드가 훨씬 직관적이고 짧아졌습니다. 나중에 모델을 Claude로 바꾸고 싶다면 ChatAnthropic()으로 객체만 바꾸면 됩니다!")

=== [1. 순수 OpenAI API 방식] ===
LangChain은 자연어 처리(NLP) 응용 프로그램을 구축하기 위한 다양한 도구와 프레임워크를 제공하는 라이브러리입니다.

=== [2. LangChain ChatOpenAI 방식] ===
LangChain은 자연어 처리 모델과 외부 데이터 소스를 연결하여 복잡한 언어 기반 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.

-> 결과: 코드가 훨씬 직관적이고 짧아졌습니다. 나중에 모델을 Claude로 바꾸고 싶다면 ChatAnthropic()으로 객체만 바꾸면 됩니다!


## 6. 비교 실험 2: 딕셔너리 vs Message 클래스
문맥(Context)을 전달할 때 기존에는 `{"role": ...}` 형태의 딕셔너리를 썼지만, LangChain은 명확한 클래스 구조를 제공합니다.

In [4]:
# 1. 기존의 딕셔너리 리스트 방식 (LangChain 최신 버전에서는 이것도 허용되긴 합니다)
dict_messages = [
    {"role": "system", "content": "당신은 해적 선장입니다. 해적 말투를 쓰세요."},
    {"role": "user", "content": "보물은 어디에 있나요?"}
]

# 2. LangChain의 객체 지향 방식 (권장)
class_messages = [
    SystemMessage(content="당신은 우아한 영국 신사입니다. 정중한 말투를 쓰세요."),
    HumanMessage(content="보물은 어디에 있나요?")
]

print("=== [딕셔너리 주입 결과] ===")
print(llm.invoke(dict_messages).content)

print("\n=== [Message 클래스 주입 결과] ===")
print(llm.invoke(class_messages).content)

print("\n-> 결과: Message 클래스를 사용하면 타입 힌트(Type Hint)를 받을 수 있고, 오타(예: 'system'을 'systme'으로 치는 실수)를 원천 차단할 수 있습니다.")

=== [딕셔너리 주입 결과] ===
아하, 보물 말이냐, 친구여! 그건 세상 끝자락의 섬에서 빛나는 금화와 보석들로 가득한 곳에 숨겨져 있다네! 하지만 그 길은 험난하고, 적들이 널 노리고 있을지도 모르지. 지도를 손에 쥐고, 내 기함에 올라타서 모험을 떠나자꾸나! 말썽꾼들, 준비는 되었느냐? 보물이 우리를 기다리고 있다네! 아아-하하하!

=== [Message 클래스 주입 결과] ===
보물에 대한 이야기는 언제나 흥미로운 주제이군요. 역사적으로 보물들은 종종 숨겨진 장소나 잃어버린 도시, 또는 유적지에서 발견되곤 했습니다. 하지만 실제로는 각 지역마다 전해지는 전설이나 신화 속 보물들이 있을 뿐입니다. 만약 특정한 보물에 대해 관심이 있으시다면, 그에 대한 정보를 더 구체적으로 말씀해 주시면 더욱 도움을 드릴 수 있을 것 같습니다. 어떤 보물을 찾고 계신지요?

-> 결과: Message 클래스를 사용하면 타입 힌트(Type Hint)를 받을 수 있고, 오타(예: 'system'을 'systme'으로 치는 실수)를 원천 차단할 수 있습니다.


## 7. 실패/주의 케이스: 모델명 오타 시뮬레이션
LangChain이 감싸고 있기 때문에, 에러가 났을 때 추적(Traceback) 과정이 다소 복잡해 보일 수 있습니다.

In [5]:
print("[잘못된 모델명 호출]")
bad_llm = ChatOpenAI(model="gpt-4o-super-max") # 존재하지 않는 모델

try:
    bad_llm.invoke("안녕?")
except Exception as e:
    print("\n🔥 에러 발생:", type(e).__name__)
    print("에러 메시지:", str(e))
    print("\n-> 주의: LangChain 내부에서 HTTP 통신을 하다가 터진 것이므로 에러 메시지에 API 에러(NotFoundError 등)가 포함되어 출력됩니다. 이를 잘 읽고 원인을 파악해야 합니다.")

[잘못된 모델명 호출]

🔥 에러 발생: NotFoundError
에러 메시지: Error code: 404 - {'error': {'message': 'The model `gpt-4o-super-max` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

-> 주의: LangChain 내부에서 HTTP 통신을 하다가 터진 것이므로 에러 메시지에 API 에러(NotFoundError 등)가 포함되어 출력됩니다. 이를 잘 읽고 원인을 파악해야 합니다.


## 8. 결과 해석

1. **추상화(Abstraction)**: `ChatOpenAI` 객체를 통해 복잡한 API 통신 로직이 `.invoke()` 메서드 하나로 추상화되었습니다. 이는 다른 회사의 LLM으로 교체할 때 코드 수정을 최소화하게 해줍니다.
2. **객체 지향 메세지**: `SystemMessage`, `HumanMessage`, `AIMessage`를 사용하면 코드의 가독성이 월등히 높아지고, 딕셔너리의 키워드 오타로 인한 런타임 버그를 예방할 수 있습니다.

## 9. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 기존에는 `client.chat.completions.create` 처럼 긴 코드를 쳤지만, LangChain의 `llm.invoke()`로 호출이 매우 단순해짐.
* LangChain을 거쳐 에러가 반환되므로 초기에는 에러 로그(`Traceback`)가 길어서 원인을 파악하기 다소 낯설게 느껴질 수 있음.

### 다음 개선 방향
* 지금은 프롬프트를 변수(`question`)로 하드코딩해서 넘기고 있음. 실제 서비스에서는 사용자의 입력이 빈칸으로 들어가는 '템플릿' 형태가 되어야 하므로 `PromptTemplate` 도입이 필요함.